In [1]:
# import torch
# print(torch.__version__)
# print(torch.cuda.is_available())


In [2]:
# import sys
# print(sys.executable)


In [3]:
import os
import cv2
import numpy as np
import torch
import torchvision
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd

from gtts import gTTS
from playsound import playsound




In [4]:
device = "xpu" if torch.xpu.is_available() else "cpu"
device


'cpu'

In [5]:
# train_dir = r"C:\All programing\python\Data Science-AIML\6 driver assistence\GTSRB\Train1"
# train_dir = r"C:\All programing\python\Data Science-AIML\6 driver assistence\GTSRB\Train"
train_dir = r"C:\All programing\python\Data Science-AIML\6 driver assistence\GTSRB\Meta2"


In [6]:
## Transformation
transform = transforms.Compose([
    transforms.Resize((64,64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])


In [7]:
## Load dataset
train_data = datasets.ImageFolder(train_dir, transform=transform)
len(train_data)

43

In [8]:
## optimize dataset
train = pd.read_csv("GTSRB/Train.csv")
train = train.sample(frac=0.5, random_state=42)   # 50% images


In [9]:
train_size = int(0.8 * len(train_data))
test_size = len(train_data) - train_size

train_dataset, test_dataset = random_split(train_data, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)

print("Train Images:", len(train_dataset))
print("Test Images:", len(test_dataset))


Train Images: 34
Test Images: 9


In [10]:

# Traffic sign classes
classes = [
"Speed limit 20","Speed limit 30","Speed limit 50","Speed limit 60","Speed limit 70",
"Speed limit 80","End Speed 80","Speed 100","Speed 120","No passing",
"No passing trucks","Right of way","Priority road","Yield","Stop","No Vehicles",
"No Trucks","No Entry","Danger","Curve Left","Curve Right","Double Curve",
"Rough Road","Slippery","Road Narrows","Construction","Traffic Signal Ahead",
"Pedestrian","Children","Bikes","Snow","Animals","End speed",
"Turn Right","Turn Left","Straight","Right Only","Left Only","Roundabout",
"End restrictions","Parking","End parking"
]
len(classes)


42

Transfer learning model

In [11]:
## Transfer Learning Model(ResNet18)
model = models.resnet18(pretrained=True)

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, 43)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

model = model.to(device)
model


c:\Users\hp\anaconda3\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\hp\anaconda3\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

Train the model

In [12]:

# Training the model
epochs = 5

for epoch in range(epochs):
    model.train()
    running_loss = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}] Loss = {running_loss/len(train_loader):.4f}")


Epoch [1/5] Loss = 5.3052
Epoch [2/5] Loss = 4.3789
Epoch [3/5] Loss = 3.4693
Epoch [4/5] Loss = 2.5993
Epoch [5/5] Loss = 1.8129


Check Accuracy

In [13]:
correct = 0
total = 0
model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Accuracy = ", 100 * correct / total, "%")


Accuracy =  100.0 %


In [14]:
### Save model
torch.save(model.state_dict(), "gtsrb_model.pth")
print("Model Saved Successfully 🎯")


Model Saved Successfully 🎯


In [15]:
### load model anytime
model.load_state_dict(torch.load("gtsrb_model.pth", map_location=device))
model.eval()


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

Real - time Driver Assistant system

In [16]:
### Voice alert
def speak(text):
    tts = gTTS(text=text, lang='en')
    tts.save("alert.mp3")
    playsound("alert.mp3")
    os.remove("alert.mp3")


In [17]:
### Preproccesing function
preprocess = transforms.Compose([
    transforms.Resize((64,64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])


In [18]:
#cap = cv2.VideoCapture(0)  # 0 → try 1,2 if USB cam


In [19]:
#frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


In [20]:
from ultralytics import YOLO
model = YOLO("yolov8n.pt")


Live Detection + Alerts

In [29]:
cap = cv2.VideoCapture(0)
prev_label = ""

model.eval()   # important

while True:
    ret, frame = cap.read()
    if not ret:
        break

    img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    img = preprocess(img).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(img)

        # -------- HANDLE EVERY POSSIBLE OUTPUT --------
        if isinstance(output, torch.Tensor):
            logits = output
        
        elif isinstance(output, (list, tuple)):
            # take first element only if tensor inside
            if torch.is_tensor(output[0]):
                logits = output[0]
            elif hasattr(output[0], "logits"):
                logits = output[0].logits
            else:
                print("Unknown list type:", type(output[0]))
                continue
        
        elif hasattr(output, "logits"):
            logits = output.logits
        
        else:
            print("Unknown output type:", type(output))
            continue

        # -------- SAFE PREDICTION --------
        probs = torch.softmax(logits, dim=1)
        conf, pred = torch.max(probs, dim=1)

        conf = conf.item()
        pred = pred.item()

        if conf < 0.7:
            label = "No Sign Detected"
        else:
            label = classes[pred]

    cv2.putText(frame, f"Detected: {label}", (40,40),
                cv2.FONT_HERSHEY_SIMPLEX,1,(0,255,0),2)

    cv2.imshow("Driver Assistance System", frame)

    if label != prev_label and label != "No Sign Detected":
        speak(label)
        prev_label = label

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()



0: 64x64 3 persons, 17.0ms
Speed: 0.0ms preprocess, 17.0ms inference, 2.4ms postprocess per image at shape (1, 3, 64, 64)
Unknown list type: <class 'ultralytics.engine.results.Results'>

0: 64x64 3 persons, 13.4ms
Speed: 0.0ms preprocess, 13.4ms inference, 1.2ms postprocess per image at shape (1, 3, 64, 64)
Unknown list type: <class 'ultralytics.engine.results.Results'>

0: 64x64 3 persons, 9.3ms
Speed: 0.0ms preprocess, 9.3ms inference, 1.3ms postprocess per image at shape (1, 3, 64, 64)
Unknown list type: <class 'ultralytics.engine.results.Results'>

0: 64x64 3 persons, 13.6ms
Speed: 0.0ms preprocess, 13.6ms inference, 2.4ms postprocess per image at shape (1, 3, 64, 64)
Unknown list type: <class 'ultralytics.engine.results.Results'>

0: 64x64 3 persons, 13.2ms
Speed: 0.0ms preprocess, 13.2ms inference, 2.0ms postprocess per image at shape (1, 3, 64, 64)
Unknown list type: <class 'ultralytics.engine.results.Results'>

0: 64x64 3 persons, 8.7ms
Speed: 0.0ms preprocess, 8.7ms inference

KeyboardInterrupt: 

In [22]:
print(ret)


True


In [ ]:
### Test Accuracy

correct = 0
total = 0
model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Accuracy = ", 100 * correct / total, "%")


Accuracy =  100.0 %


In [ ]:
#pip install opencv-python mediapipe ultralytics pyttsx3 numpy


   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   --- ------------------------------------ 0.8/10.4 MB 4.9 MB/s eta 0:00:02
   ---- ----------------------------------- 1.0/10.4 MB 2.2 MB/s eta 0:00:05
   ----- ---------------------------------- 1.3/10.4 MB 2.3 MB/s eta 0:00:04
   ---------- ----------------------------- 2.6/10.4 MB 2.9 MB/s eta 0:00:03
   ------------- -------------------------- 3.4/10.4 MB 3.0 MB/s eta 0:00:03
   ---------------- ----------------------- 4.2/10.4 MB 3.2 MB/s eta 0:00:02
   ------------------- -------------------- 5.0/10.4 MB 3.5 MB/s eta 0:00:02
   --------------------- ------------------ 5.5/10.4 MB 3.3 MB/s eta 0:00:02
   ----------------------- ---------------- 6.0/10.4 MB 3.2 MB/s eta 0:00:02
   ----------------------- ---------------- 6.0/10.4 MB 3.2 MB/s eta 0:00:02
   -----------------

In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8n.pt")


Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\hp\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
# open webcam
# loop:
#     read frame

#     # 1 DROWSINESS
#     detect face + eyes
#     calculate eye openness
#     if eyes closed long:
#         speak("Wake up driver")

#     # 2 YAWNING
#     detect mouth opening
#     if big yawning:
#         speak("Do not yawn while driving")

#     # 3 TRAFFIC SIGN
#     run YOLO
#     if sign detected:
#         speak("Speed Limit 20")
    
#     show frame


SyntaxError: invalid syntax (3247110703.py, line 1)